## Tool definition

### Environment Variable Initialization
This cell utilizes `load_dotenv()` from the `dotenv` library. It loads key-value pairs from a local `.env` file into the environment variables of the current process, ensuring that sensitive API credentials required by LangChain or external tools (like Tavily) are securely accessed without exposing them in the code.

In [12]:
from dotenv import load_dotenv
load_dotenv()

True

### Defining the 'square_root' Tool
Here, we register a custom function `tool1` as a tool named "square_root" using the `@tool` decorator. The docstring `"""Returns the square root of a number."""` is programmatically extracted by the LangChain agent to understand the tool's purpose, allowing the model to determine when this specific function is appropriate to call.

In [13]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Returns the square root of a number."""
    return x ** 0.5

### Refined Tool Definition
This cell serves as an alternative or update to the previous definition. By providing a explicit `description` parameter in the `@tool` decorator, we provide more granular guidance to the LLM. This explicitly instructs the agent that this tool is for calculating the square root of a number, reinforcing the model's decision-making logic when a user asks for such operations.

In [14]:
@tool("square_root")
def tool1(x: float) -> float:
    """Returns the square root of a number."""
    return x ** 0.5

In [15]:
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    """Returns the square root of a number."""
    return x ** 0.5

### Tool Invocation
In this cell, we manually trigger the `tool1` function by calling its `invoke` method with a dictionary containing the required argument `x: 425`. This demonstrates how to interact with a tool directly, bypassing the agent's logic to verify that the function behaves as expected and produces the correct numerical output.

In [16]:
tool1.invoke({"x": 425})

20.615528128088304

## Adding to agents

### Creating the Arithmetic Agent
This cell initializes the `agent` using `create_agent`. We specify the model `"gpt-5-nano"` and assign it a `system_prompt` that defines its persona as an "arithmetic wizard." The `tools` list is populated with `[tool1]`, granting the agent the capability to perform square root calculations autonomously when processing the `HumanMessage`.

In [17]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent= create_agent(
    "gpt-5-nano",
    system_prompt= "You are an arithmetic wizard. Use your tools to calculate the square root and square of any number.",
    tools=[tool1]
)

question= HumanMessage(content="What is the square root of 426?")

response= agent.invoke({"messages":[question]})

print(response["messages"][-1].content)

Approximately 20.639767440550294.

Note: 426 is not a perfect square, so its square root is irrational. If you’d like a rounded value, I can provide it to any precision (e.g., 20.639767). Want me to also square the result for verification?


### Inspecting the Agent's Execution
By using `pprint(response["messages"])`, we print the full list of messages generated during the agent's turn. This provides a detailed view of the conversation history, showcasing the `HumanMessage` (request), `AIMessage` (the agent's plan to call a tool), and `ToolMessage` (the tool's output), which is essential for debugging the agent's reasoning process.

In [18]:
from pprint import pprint

pprint(response["messages"])

[HumanMessage(content='What is the square root of 426?', additional_kwargs={}, response_metadata={}, id='acc1372c-0f93-4801-88a9-1d379cbfc266'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2139, 'prompt_tokens': 158, 'total_tokens': 2297, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2112, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dpd9J25MXG3RHOiI3PfJHfOlq2f3S', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb7a6-4a20-76f3-b108-ec81b2a57e30-0', tool_calls=[{'name': 'square_root', 'args': {'x': 426}, 'id': 'call_Ku17I4EJpl6SjodzLgG3k1sf', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 2139, 'total

### Extracting Tool Call Metadata
This cell targets the second element in the messages list (`response["messages"][1]`) to specifically print the `tool_calls` attribute. This output is critical as it reveals the structured data format the LLM generates when it decides to use a tool, including the function `name`, the input `args`, and a unique `id` for tracking.

In [19]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 426}, 'id': 'call_Ku17I4EJpl6SjodzLgG3k1sf', 'type': 'tool_call'}]


## Without web search

### Testing Agent Knowledge Boundaries
By querying the agent with "How up to date is your training knowledge?" without providing a search tool, we intentionally trigger the agent's base limitations. The response demonstrates that the agent is restricted to its pre-training data (November 2023) and cannot access real-time information, establishing the "before" state for our next experiment.

In [20]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent= create_agent(
    "gpt-5-nano"
)

question= HumanMessage(content="How up to date is your training knowledge?")

response= agent.invoke({
    "messages":[question]
}
)

print(response["messages"][-1].content)

Knowledge cutoff is around November 2023. I don’t have live internet access by default, so I can’t pull in new events or data after that unless a browsing tool is enabled for me. If you want the latest information, I can try to fetch it with browsing (if available) or you can share sources and I’ll summarize them.


## Add web search tool

### Implementing Web Search Tooling
This cell sets up a new `web_search` tool by wrapping the `tavily_client.search` method in the `@tool` decorator. By defining this tool, we enable the agent to bridge the knowledge gap identified in cell [20], allowing it to query external information in real-time when the internal model knowledge is insufficient.

### Testing the Search Tool
Here, we execute `web_search.invoke()` with a specific query about current events ("Who is the current mayor of cairo..."). This verifies that the `TavilyClient` correctly connects to the internet and returns structured data (including URLs and titles) that the agent can later parse to answer dynamic, non-static user questions.

In [24]:
from langchain.tools import tool
from typing import Any, Dict
from tavily import TavilyClient

travily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
     """Search the web for information"""
     return travily_client.search(query)
web_search.invoke({"query": "Who is the current mayor of cairo in egypt?"})

{'query': 'Who is the current mayor of cairo in egypt?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://timesenterprise.com/2025/06/19/ed-robinson-elected-as-cairo-mayor',
   'title': 'Ed Robinson elected as Cairo Mayor | Thomasville Times-Enterprise',
   'content': 'Edgar M. Robinson defeated Shenkia Johnson and Silvia Salgado in a called special election on Tuesday to take the Mayoral seat.',
   'score': 0.78788257,
   'raw_content': None},
  {'url': 'https://ballotpedia.org/Arlisha_L._Williams_(Mayor_of_Cairo,_Georgia,_candidate_2025)',
   'title': 'Arlisha L. Williams (Mayor of Cairo, Georgia, candidate 2025)',
   'content': 'General election for Mayor of Cairo. Incumbent Edgar M. Robinson and Arlisha L. Williams ran in the general election for Mayor of Cairo on November 4, 2025.',
   'score': 0.77096564,
   'raw_content': None},
  {'url': 'https://www.wctv.tv/2026/01/13/city-cairo-makes-history-with-swearing-first-ever-woman-mayor',
   

### Agent-Integrated Search
We re-initialize the agent, this time providing it with the `web_search` tool. When asked about a real-time event ("Who is the current Governor of cairo..."), the agent now has the agency to use this tool. It successfully retrieves current, accurate data, showcasing the transition from a static model to a functional, informed agent.

In [26]:
agent= create_agent(
    model= "gpt-5-nano",
    tools=[web_search]
)

question= HumanMessage(content="Who is the current Governor  of cairo in egypt?")

response= agent.invoke({"messages":[question]})

print(response["messages"][-1].content)

Ibrahim Saber Khalil (often referred to as Ibrahim Saber) is the current Governor of Cairo Governorate in Egypt.

Sources: Cairo Governorate official page lists Ibrahim Saber Khalil as Governor; EgyptToday also identifies him as the Governor of Cairo. If you need real-time confirmation, you can check the official Cairo Governorate site or recent credible news.


### Final Response Inspection
We use `pprint` on the final `response["messages"]` to visualize the completed chain of thought. We can observe how the agent successfully utilized the `ToolMessage` result from the search engine to synthesize the final answer provided to the user, proving the effectiveness of the integrated RAG-like (Retrieval-Augmented Generation) workflow.

In [27]:
pprint(response["messages"])

[HumanMessage(content='Who is the current Governor  of cairo in egypt?', additional_kwargs={}, response_metadata={}, id='21cb7543-d061-4c76-b9dd-7da31efb2fe1'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 135, 'total_tokens': 354, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpdQv1Kr5BmprZiYiooCSEygoBzWt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019eb7b6-f2ba-77e3-a3de-f7fa075ef549-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current Governor of Cairo Egypt'}, 'id': 'call_tbkZPG88A0cAA3kG0rMLny2I', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'inp